In [5]:
%matplotlib widget
import importlib
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import pandas as pd
import io


os.chdir(r'C:\Users\nberr\ENS_Lyon\Thèse\Code\Optical detection\Permittivity changes estimation')
from Permittivity_change_func import *
import Permittivity_change_func
importlib.reload(Permittivity_change_func)
from Permittivity_change_func  import *


fontsize=30
import matplotlib as mpl
mpl.rcParams["font.size"] = 18

## Calculation of permittivity change with acoustic modes

Here, we focus on the contribution to $\sigma_{ext}(\lambda)$ :
$$\Delta \sigma_{ext}(\lambda) = \frac{\partial \sigma_{ext}}{\partial \varepsilon_1} \Delta \varepsilon_1 + \frac{\partial \sigma_{ext}}{\partial \varepsilon_2} \Delta \varepsilon_2$$

And then : 
$$ \Delta \varepsilon_1(\lambda) = \left( a^D(\lambda) + a^{ib}(\lambda) \right) \left( \frac{dV}{V} \right)$$
$$ \Delta \varepsilon_2(\lambda) = \left( b^D(\lambda) + b^{ib}(\lambda) \right) \left( \frac{dV}{V} \right)$$

In [6]:
ND_names = ['Sapphire 2,3 - NDd zone 3', 'Silice thick 4,5 - D1 zone1', 'Membrane 3,5 - ND3,5-5']

global_dir_root = r'H:\commun\Experiments (from 2018)\Nanoscope 2\Projects\ULTRASINGLE\Measurements\AuND_specific_AR_Marguet\Optical study'

All_data = {}


## Get Plasmonic derivatives

For each ND, we made the calculation on comsol : 

- $\sigma_{ext}(\lambda, \varepsilon(\lambda))$
- $\sigma_{ext}(\lambda, \varepsilon(\lambda) + d\varepsilon_1(\lambda))$
- $\sigma_{ext}(\lambda, \varepsilon(\lambda) + d\varepsilon_2(\lambda))$

I set the values $d\varepsilon_i(\lambda) = 0.1 \cdot \varepsilon_i(\lambda)$

Then I deduce the plasmonic derivatives : $$ \frac{\partial \sigma_{ext}}{\partial \varepsilon_i}(\lambda) = \frac{\sigma_{ext}(\lambda, \varepsilon(\lambda) + d\varepsilon_i(\lambda)) - \sigma_{ext}(\lambda, \varepsilon(\lambda))}{d\varepsilon_i (\lambda)} $$ 

In [7]:
for key in ND_names :
    print(key)
    All_data[key] = {}
    dir_plasmonic = '\\'.join((global_dir_root, key, 'Optical sensitivity calculation'))
    All_data[key]['plasmonic_derivative'] = get_plasmonic_derivatives(dir_plasmonic, 'plasmonic_derivatives.csv')

Sapphire 2,3 - NDd zone 3
Silice thick 4,5 - D1 zone1
Membrane 3,5 - ND3,5-5


## Get Drude sensitivity

In [8]:
#Gold values from Romain PhD manuscript
w_p = 9.01 * sc.constants.e / sc.constants.hbar 
gamma_0 = 70*1e-3 * sc.constants.e / sc.constants.hbar

def eps_1_drude(w, gamma=gamma_0, w_p=w_p) :
    return(1 - (w_p**2)/(w**2+gamma**2))
def eps_2_drude(w, gamma=gamma_0, w_p=w_p) :
    return((gamma*w_p**2)/(w*(w**2 + gamma**2)))

# in rad/s
def a_drude_rad(w) :
    return (1- eps_1_drude(w))
def b_drude_rad(w) :
    return(-eps_2_drude(w))

# in eV
def a_drude_ernergy(energy) :
    return (1- eps_1_drude(energy* sc.constants.e / sc.constants.hbar  ))
def b_drude_ernergy(energy) :
    return(-eps_2_drude(energy * sc.constants.e / sc.constants.hbar ))

# in meters
def a_drude_wavelength(lda) :
    return (1- eps_1_drude(2*np.pi*sc.constants.c /lda))
def b_drude_wavelength(lda) :
    return(-eps_2_drude(2*np.pi*sc.constants.c /lda))

## Get interband sensitivity